# Build a RAG System Using LangChain

## 1. Project Overview

The objective of this project is to build a Retrieval-Augmented Generation (RAG) system using LangChain. The system answers questions based on the TechNova Solutions employee policy document rather than relying only on the language model's pre-trained knowledge.

## 2. Objectives

- Load and process the employee policy from a `.txt` file.
- Split the document into smaller text chunks.
- Generate embeddings using a HuggingFace sentence-transformer model.
- Store embeddings in a FAISS vector database.
- Retrieve relevant document chunks for a user's question.
- Use a Groq-hosted Large Language Model (LLM) to generate answers based on the retrieved context.
- Avoid hallucinations by instructing the model not to invent information missing from the document.

## 3. Technologies Used

- **Python:** Implementation
- **LangChain:** RAG pipeline and document processing
- **Groq:** Large Language Model inference
- **HuggingFace:** Text embeddings
- **FAISS:** Vector storage and similarity search
- **Google Colab:** Development and execution environment

## 4. Implementation Workflow

1. Install the required Python libraries.
2. Configure the Groq API key and initialize the language model.
3. Load the TechNova employee policy `.txt` file.
4. Split the document using `RecursiveCharacterTextSplitter`.
5. Generate vector embeddings for the text chunks.
6. Store the embeddings in a FAISS vector database.
7. Configure a retriever to find relevant policy information.
8. Create a RAG chain that passes retrieved context to the language model.
9. Run the assignment questions and examine the generated answers.

## 5. Assignment Questions

**Q1.** How many paid annual leave days does a full-time employee receive per year?

**Q2.** What do the policies say about working remotely on Wednesday and taking leave on Friday?

**Q3.** What is the parental leave policy?

## 6. Expected Results

- **Q1:** The policy states that full-time employees receive 18 paid annual leave days per year.
- **Q2:** Remote work is permitted on Wednesdays and Fridays, subject to the policy's email notice and manager-approval requirements. Annual leave on Friday requires a leave request at least three calendar days in advance and manager confirmation.
- **Q3:** If parental leave information is absent from the provided policy document, the system should explicitly state that the document does not specify it rather than inventing a policy.

## 7. Conclusion

This project demonstrates how Retrieval-Augmented Generation can connect a language model to a specific document. By retrieving relevant text before generating a response, the system can provide more context-grounded answers and identify information that is missing from the source document.

# Hands-on Assignment: Build a RAG System Using LangChain
**TechNova Solutions Employee Policy**



In [1]:
!pip install -q langchain langchain-core langchain-community langchain-text-splitters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
!pip install -q langchain-groq langchain-huggingface sentence-transformers faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.5 MB/s eta 0:00:00


## 1. Set up the language model

Add `GROQ_API_KEY` in Google Colab's **Secrets** (key icon on the left) and allow notebook access.

In [3]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=userdata.get("GROQ_API_KEY"),
)


In [4]:
# Check that the model is working
response = llm.invoke("What is LangChain? Answer briefly.")
response.content


'LangChain is an open‑source framework that helps developers build applications powered by large language models (LLMs). It provides tools for chaining together prompts, managing memory, integrating external data sources, and creating agents that can reason and act autonomously.'

## 2. Load the provided policy `.txt` file



In [5]:
# Optional Document example (not used by the RAG pipeline)
from langchain_core.documents import Document

doc1 = Document(
    page_content="Example document only.",
    metadata={"source": "example"}
)


In [6]:
doc1

Document(metadata={'source': 'example'}, page_content='Example document only.')

In [7]:
import os
from google.colab import files
from langchain_community.document_loaders import TextLoader

# Use technova_policy.txt if it already exists; otherwise upload your policy .txt file.
txt_file = "technova_policy.txt"

if not os.path.exists(txt_file):
    print("Upload your TechNova policy .txt file:")
    uploaded = files.upload()
    txt_files = [name for name in uploaded.keys() if name.lower().endswith(".txt")]
    if not txt_files:
        raise FileNotFoundError("Please upload a .txt policy file and run this cell again.")
    txt_file = txt_files[0]

loader = TextLoader(txt_file, encoding="utf-8")
docs1 = loader.load()

print("Loaded file:", txt_file)
print("Number of documents:", len(docs1))
print(docs1[0].page_content[:1000])


Loaded file: technova_policy.txt
Number of documents: 1
﻿TechNova Solutions - Employee Policy Document


1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation.


2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.


3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.


4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for more than two consecutive working days, a medical certificate must be submitted to HR.


5. Learning Allowance: Each full-time employee can claim up to ₹12,000 per 

/tmp/ipykernel_808/1823633964.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## 3. Split the document into chunks


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 350, chunk_overlap = 50)

In [9]:
chunks = splitter.split_documents(docs1)

In [10]:
chunks

[Document(metadata={'source': 'technova_policy.txt'}, page_content="\ufeffTechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation."),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.\n\n\n3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.'),
 Document(metadata={'source': 'technova_policy.txt'}, page_content='4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for mor

In [11]:
print(splitter._chunk_size)
print(splitter._chunk_overlap)

350
50


In [12]:
print("Number of chunks:", len(chunks))

Number of chunks: 4


## 4. Create embeddings


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
vectors = embedding_model.embed_documents([doc.page_content for doc in chunks])

In [15]:
len(vectors)

4

In [16]:
len(vectors[0])

384

## 5. Store the embeddings in FAISS


In [17]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embedding_model)


In [18]:
vectorstore

In [19]:
# Example similarity search

retrieved_docs = vectorstore.similarity_search("How many annual leave days are provided?",k=3)
retrieved_docs


[Document(id='5b343479-ecf4-4685-b1fc-7843a7d5856b', metadata={'source': 'technova_policy.txt'}, page_content="\ufeffTechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation."),
 Document(id='2cee5929-6b1b-4816-ba2f-37b29d969be7', metadata={'source': 'technova_policy.txt'}, page_content='4. Sick Leave: Employees receive 10 paid sick leave days per calendar year. If an employee is absent due to illness for more than two consecutive working days, a medical certificate must be submitted to HR.'),
 Document(id='2f48750d-d7aa-48d8-89f5-783b3cfb374b', metadata={'source': 'technova_policy.txt'}, page_content='2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least 

## 6. Create a retriever


In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [21]:
retriever.invoke("What are the work from home rules?")

[Document(id='2f48750d-d7aa-48d8-89f5-783b3cfb374b', metadata={'source': 'technova_policy.txt'}, page_content='2. Work From Home: Employees may work remotely on Wednesdays and Fridays. Employees must inform their manager by email at least one working day in advance. Remote work is subject to manager approval.\n\n\n3. Working Hours: The standard working day is from 9:30 AM to 5:30 PM, Monday through Friday. Employees receive a one-hour lunch break.'),
 Document(id='5b343479-ecf4-4685-b1fc-7843a7d5856b', metadata={'source': 'technova_policy.txt'}, page_content="\ufeffTechNova Solutions - Employee Policy Document\n\n\n1. Annual Leave: Every full-time employee receives 18 paid annual leave days per calendar year. Employees must submit annual leave requests to their manager at least 3 calendar days before the requested leave date. Leave approval depends on the manager's confirmation."),
 Document(id='c93e3d16-dcc4-49b1-8da7-d6ef702bf0c8', metadata={'source': 'technova_policy.txt'}, page_con

## 7. Create the RAG chain


In [22]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


In [23]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [24]:
template = """
Use only the information in the context below to answer the question.
Keep the answer concise and factual.
If the policy document does not contain the requested information, say:
"The provided policy document does not specify this information."
Do not guess or add policy details that are not in the context.

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate.from_template(template)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 8. Run the three assignment questions


In [25]:
# Question 1
query1 = "How many paid annual leave days does a full-time employee receive per year?"
answer1 = rag_chain.invoke(query1)
print("Q1:", query1)
print("Answer:", answer1)


Q1: How many paid annual leave days does a full-time employee receive per year?
Answer: A full-time employee receives **18 paid annual leave days per year**.


In [26]:
# Question 2
query2 = "An employee wants to work remotely on Wednesday and take leave on Friday. What do the policies say?"
answer2 = rag_chain.invoke(query2)
print("Q2:", query2)
print("Answer:", answer2)


Q2: An employee wants to work remotely on Wednesday and take leave on Friday. What do the policies say?
Answer: The policies allow both actions, but each requires its own notice and manager approval:

* **Remote work on Wednesday** – Employees may work from home on Wednesdays (or Fridays) if they email their manager at least one working day in advance and obtain the manager’s approval.

* **Leave on Friday** – Annual leave must be requested of the manager at least 3 calendar days before the intended leave date and is granted only after the manager confirms it.


In [27]:
# Question 3
query3 = "What is the parental leave policy?"
answer3 = rag_chain.invoke(query3)
print("Q3:", query3)
print("Answer:", answer3)


Q3: What is the parental leave policy?
Answer: The provided policy document does not specify this information.


## Expected answer checks

- **Q1:** 18 paid annual leave days per calendar year.
- **Q2:** Wednesday remote work requires emailing the manager at least one working day in advance and manager approval. Friday leave should be requested at least 3 calendar days in advance and depends on manager confirmation.
- **Q3:** The provided policy document does not specify parental leave.

